# Week 28, Day 2: A tour of the brain: areas, layers and pathways

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w028/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: A signal flowing through the layers

Model four layers as leaky rate units (τ = 10 ms) connected in the canonical order: thalamus to L4, L4 to L2/3, L2/3 to L5, L5 to L6 (weights 1), with a brief thalamic pulse from 5 to 10 ms. Simulate 100 ms with Euler steps of 0.1 ms, store the rates in `r` (layers L4, L2/3, L5, L6) and each layer's peak time in `peak`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
names = ["L4", "L2/3", "L5", "L6"]
W = np.array([[0, 0, 0, 0],      # row = target, column = source
              [1, 0, 0, 0],
              [0, 1, 0, 0],
              [0, 0, 1, 0]], float)
tau, dt = 10.0, 0.1
ts = np.arange(0, 100, dt)
thal = ((ts >= 5) & (ts < 10)).astype(float)
r = np.zeros((4, len(ts)))
for k in range(1, len(ts)):
    inp = W @ r[:, k - 1] + np.array([thal[k - 1], 0, 0, 0])
    r[:, k] = ...                                 # leaky integration towards the input
peak = {n: float(ts[np.argmax(r[i])]) for i, n in enumerate(names)}
print("peak times (ms):", peak)
for i, n in enumerate(names): plt.plot(ts, r[i], label=n)
plt.legend(); plt.xlabel("ms"); plt.show()

In [ ]:
import numpy as np
def test_order():
    "the response peaks in layer 4 first, then layers 2/3, then layer 5"
    assert peak["L4"] < peak["L2/3"] < peak["L5"]

def test_shape():
    "r holds one row per layer and one column per time step"
    assert r.shape == (4, len(ts))

test_order()
test_shape()
print("All checks passed")

### Exercise: The visual hierarchy as a graph

Compute the number of feedforward steps from V1 to every area, `dist`, by breadth-first search, and the fraction of feedforward connections matched by a feedback connection in the opposite direction, `reciprocity`.

In [ ]:
from collections import deque
feedforward = [("V1", "V2"), ("V1", "MT"), ("V2", "V4"), ("V2", "MT"), ("V4", "TEO"), ("TEO", "TE"),
               ("V4", "TE"), ("MT", "MST"), ("MST", "7a"), ("V4", "MT")]
feedback = [("V2", "V1"), ("MT", "V1"), ("V4", "V2"), ("MT", "V2"), ("TEO", "V4"), ("TE", "TEO"),
            ("MST", "MT"), ("7a", "MST")]
graph = {}
for a, b in feedforward: graph.setdefault(a, []).append(b)

dist, queue = {"V1": 0}, deque(["V1"])
while queue:
    a = queue.popleft()
    for b in graph.get(a, []):
        if b not in dist:
            dist[b] = ...
            queue.append(b)
reciprocity = ...
print("feedforward steps from V1:", dist); print(f"reciprocity {reciprocity:.2f}")

In [ ]:
import numpy as np
def test_dist():
    "dist holds breadth-first distances from V1 along feedforward edges"
    assert dist["V1"] == 0 and dist["V2"] == 1 and dist["TE"] == 3 and dist["7a"] == 3

def test_recip():
    "reciprocity is the fraction of feedforward edges whose reverse is a feedback edge"
    assert abs(reciprocity - sum((b, a) in feedback for a, b in feedforward) / len(feedforward)) < 1e-12

test_dist()
test_recip()
print("All checks passed")

### Exercise: The exponential distance rule

For 300 pairs of areas, connection strength is 10 exp(−λd) times log-normal scatter (sd 0.8 in natural log), with λ = 0.19 per mm and distances 2 to 40 mm (local rng seed 13). Fit log strength against distance by least squares to estimate `lam_hat`, and compute the distance over which strength halves, `half_dist`.

In [ ]:
r13 = np.random.default_rng(13)
d = r13.uniform(2, 40, 300)
strength = 10 * np.exp(-0.19 * d) * np.exp(0.8 * r13.normal(size=300))
slope, intercept = ...
lam_hat = -slope
half_dist = ...
print(f"decay rate {lam_hat:.3f} per mm;  strength halves every {half_dist:.1f} mm")
plt.semilogy(d, strength, ".", ms=3); plt.xlabel("distance (mm)"); plt.ylabel("connection strength"); plt.show()

In [ ]:
import numpy as np
def test_fit():
    "lam_hat is the decay rate from regressing log strength on distance, within 15% of the true 0.19 per mm"
    assert abs(lam_hat - 0.19) / 0.19 < 0.15

def test_half():
    "half_dist is ln 2 / lam_hat"
    assert abs(half_dist - np.log(2) / lam_hat) < 1e-12

test_fit()
test_half()
print("All checks passed")